In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import shutil

shutil.copy('/content/drive/MyDrive/dataPS.zip', '/content/dataset.zip')

'/content/dataset.zip'

In [4]:
import zipfile

with zipfile.ZipFile('/content/dataset.zip', 'r') as zip_ref:
    zip_ref.extractall('/content')

In [5]:
import os
os.listdir('/content')

['.config',
 'GAF_512',
 'GAF_1024',
 'dataset.zip',
 'drive',
 'CSV',
 'sample_data']

In [6]:
#  CHARGEMENT DES BASES NORMALISÉES (depuis le notebook précédent)
import pandas as pd
import numpy as np

# Si vous avez les CSV sauvegardés :
try:
    df_normal_norm = pd.read_csv("/content/CSV/Normalises/Base_1D_Normal_normalized.csv")
    df_irf_norm    = pd.read_csv("/content/CSV/Normalises/Base_1D_IRF_normalized.csv")
    df_orf_norm    = pd.read_csv("/content/CSV/Normalises/Base_1D_ORF_normalized.csv")
    print(f" Données chargées — Normal: {df_normal_norm.shape}, IRF: {df_irf_norm.shape}, ORF: {df_orf_norm.shape}")
except FileNotFoundError:
    print(" CSV introuvables. Assurez-vous d'avoir exécuté le notebook précédent.")
    print("   Les fichiers attendus : Base_1D_Normal_normalized.csv, Base_1D_IRF_normalized.csv, Base_1D_ORF_Final_normalized.csv")

✅ Données chargées — Normal: (100, 5000), IRF: (100, 5000), ORF: (100, 5000)


---
# 🧠 Section 7.1 — Implémentation du Réseau Siamois Conv1D
### Approche basée sur les signaux temporels
---

In [7]:
# ─────────────────────────────────────────────────────────────
#  IMPORTS GLOBAUX — version hybride Conv1D + Conv2D
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.layers import (
    Input, Conv1D, Conv2D, MaxPooling1D, MaxPooling2D,
    Dense, Flatten, BatchNormalization, Dropout,
    GlobalAveragePooling1D, GlobalAveragePooling2D,
    Concatenate, Lambda, SpatialDropout1D, Activation
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

from scipy.signal import spectrogram as scipy_spectrogram
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix,
    roc_curve, auc, classification_report
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.manifold import TSNE

# Reproductibilité
np.random.seed(42)
tf.random.set_seed(42)

print(f" TensorFlow {tf.__version__} chargé")
print(f" GPU disponible : {len(tf.config.list_physical_devices('GPU')) > 0}")


✅ TensorFlow 2.20.0 chargé
✅ GPU disponible : False


In [8]:
#  PARAMÈTRES GLOBAUX


N_FEATURES  = 512
EMBED_DIM   = 64
MARGIN      = 1.5
N_PAIRS     = 8000
BATCH_SIZE  = 32
EPOCHS      = 50
DROP_RATE   = 0.4
L2_REG      = 1e-4

# Spectrogramme
FS          = 10000
NPERSEG     = 64
NOVERLAP    = 32

#  PRÉPARATION DES DONNÉES BRUTES

X_normal = df_normal_norm.values[:, :N_FEATURES].astype(np.float32)
X_irf    = df_irf_norm.values[:,    :N_FEATURES].astype(np.float32)
X_orf    = df_orf_norm.values[:,    :N_FEATURES].astype(np.float32)

labels_normal = np.zeros(len(X_normal), dtype=int)
labels_irf    = np.ones( len(X_irf),   dtype=int)
labels_orf    = np.full( len(X_orf), 2, dtype=int)

X_all = np.concatenate([X_normal, X_irf, X_orf], axis=0)
y_all = np.concatenate([labels_normal, labels_irf, labels_orf])

print(f" Données agrégées : X={X_all.shape}, y={y_all.shape}")
print(f"   Normal={np.sum(y_all==0)}, IRF={np.sum(y_all==1)}, ORF={np.sum(y_all==2)}")


✅ Données agrégées : X=(300, 512), y=(300,)
   Normal=100, IRF=100, ORF=100


In [9]:
#  TRANSFORMATION SPECTROGRAMME
def compute_spectrogram(signal, fs=FS, nperseg=NPERSEG, noverlap=NOVERLAP):
    """Calcule le spectrogramme log-normalisé d'un signal 1D.

    Retourne un tableau 2D (freq_bins, time_bins) normalisé dans [0,1].
    Utilise log1p pour compresser la dynamique (équivalent dB).
    """
    f, t, Sxx = scipy_spectrogram(signal, fs=fs, nperseg=nperseg, noverlap=noverlap)
    Sxx = np.log1p(Sxx)           # compression logarithmique
    max_val = np.max(Sxx)
    if max_val > 0:
        Sxx = Sxx / max_val       # normalisation dans [0, 1]
    return Sxx.astype(np.float32)

def transform_to_spectrograms(X):
    """Transforme un batch de signaux 1D en spectrogrammes 2D.

    Entrée  : X de forme (N, T)
    Sortie  : (N, freq_bins, time_bins, 1)  — canal ajouté pour Conv2D
    """
    specs = [compute_spectrogram(x) for x in X]
    specs = np.array(specs)       # (N, freq_bins, time_bins)
    return specs[..., np.newaxis] # (N, freq_bins, time_bins, 1)

# Test de la transformation sur un signal exemple
spec_test = compute_spectrogram(X_normal[0])
print(f" Spectrogramme d'un signal :")
print(f"   Signal d'entrée  : ({N_FEATURES},)")
print(f"   Spectrogramme    : {spec_test.shape}  (freq_bins × time_bins)")

# Vérifier la forme pour tout le dataset
specs_sample = transform_to_spectrograms(X_normal[:3])
SPEC_SHAPE = specs_sample.shape[1:]  # (freq_bins, time_bins, 1)
print(f"   Forme batch      : {specs_sample.shape}")
print(f"   SPEC_SHAPE (input Conv2D) : {SPEC_SHAPE}")

# Visualisation d'un spectrogramme par classe
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (X_cls, lbl) in zip(axes, [(X_normal,'Normal'),(X_irf,'IRF'),(X_orf,'ORF')]):
    spec = compute_spectrogram(X_cls[0])
    im = ax.imshow(spec, aspect='auto', origin='lower', cmap='viridis')
    ax.set_title(f'Spectrogramme — {lbl}', fontweight='bold')
    ax.set_xlabel('Trames temporelles'); ax.set_ylabel('Bins fréquentiels')
    plt.colorbar(im, ax=ax, label='Amplitude (log)')
plt.suptitle('Représentation Fréquentielle (STFT log-normalisé) par Classe',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('spectrogrammes_classes.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Spectrogrammes visualisés et sauvegardés.")


✅ Spectrogramme d'un signal :
   Signal d'entrée  : (512,)
   Spectrogramme    : (33, 15)  (freq_bins × time_bins)
   Forme batch      : (3, 33, 15, 1)
   SPEC_SHAPE (input Conv2D) : (33, 15, 1)
✅ Spectrogrammes visualisés et sauvegardés.


In [10]:
#  SÉPARATION STRICTE SIGNAUX → PUIS PAIRES (anti-data-leakage)
np.random.seed(42)

def split_class(X_cls, test_size=0.20, val_size=0.15):
    idx = np.arange(len(X_cls))
    idx_tv, idx_test = train_test_split(idx, test_size=test_size, random_state=42)
    ratio_val = val_size / (1 - test_size)
    idx_train, idx_val = train_test_split(idx_tv, test_size=ratio_val, random_state=42)
    return idx_train, idx_val, idx_test

idx_n_tr, idx_n_va, idx_n_te = split_class(X_normal)
idx_i_tr, idx_i_va, idx_i_te = split_class(X_irf)
idx_o_tr, idx_o_va, idx_o_te = split_class(X_orf)

X_sig_train = np.concatenate([X_normal[idx_n_tr], X_irf[idx_i_tr], X_orf[idx_o_tr]])
y_sig_train = np.concatenate([np.zeros(len(idx_n_tr)), np.ones(len(idx_i_tr)), np.full(len(idx_o_tr),2)]).astype(int)
X_sig_val   = np.concatenate([X_normal[idx_n_va], X_irf[idx_i_va], X_orf[idx_o_va]])
y_sig_val   = np.concatenate([np.zeros(len(idx_n_va)), np.ones(len(idx_i_va)), np.full(len(idx_o_va),2)]).astype(int)
X_sig_test  = np.concatenate([X_normal[idx_n_te], X_irf[idx_i_te], X_orf[idx_o_te]])
y_sig_test  = np.concatenate([np.zeros(len(idx_n_te)), np.ones(len(idx_i_te)), np.full(len(idx_o_te),2)]).astype(int)

print(f"Signaux — Train: {X_sig_train.shape[0]} | Val: {X_sig_val.shape[0]} | Test: {X_sig_test.shape[0]}")

#  AUGMENTATION DE DONNÉES
def augment_signal(x, noise_std=0.008, shift_max=40, scale_range=(0.95, 1.05)):
    """Augmentation : bruit gaussien + décalage temporel + scaling d'amplitude."""
    x = x + np.random.normal(0, noise_std, x.shape).astype(np.float32)
    shift = np.random.randint(-shift_max, shift_max)
    x = np.roll(x, shift)
    scale = np.random.uniform(*scale_range)
    x = x * scale
    return np.clip(x, 0.0, 1.0)

#  CRÉATION DE PAIRES
def create_pairs_from_split(X, y, n_pairs, augment=False, seed=42):
    np.random.seed(seed)
    classes = np.unique(y)
    idx_by_class = {c: np.where(y == c)[0] for c in classes}
    pairs_A, pairs_B, pair_labels = [], [], []
    n_pos = n_pairs // 2

    for _ in range(n_pos):
        cls = np.random.choice(classes)
        idxs = idx_by_class[cls]
        if len(idxs) < 2: continue
        ia, ib = np.random.choice(idxs, 2, replace=False)
        xa = augment_signal(X[ia].copy()) if augment else X[ia].copy()
        xb = augment_signal(X[ib].copy()) if augment else X[ib].copy()
        pairs_A.append(xa); pairs_B.append(xb); pair_labels.append(0)

    for _ in range(n_pairs - n_pos):
        ca, cb = np.random.choice(classes, 2, replace=False)
        ia = np.random.choice(idx_by_class[ca])
        ib = np.random.choice(idx_by_class[cb])
        xa = augment_signal(X[ia].copy()) if augment else X[ia].copy()
        xb = augment_signal(X[ib].copy()) if augment else X[ib].copy()
        pairs_A.append(xa); pairs_B.append(xb); pair_labels.append(1)

    perm = np.random.permutation(len(pairs_A))
    return (np.array(pairs_A)[perm].astype(np.float32),
            np.array(pairs_B)[perm].astype(np.float32),
            np.array(pair_labels)[perm].astype(np.float32))

n_tr = N_PAIRS
n_va = N_PAIRS // 4
n_te = N_PAIRS // 4

A_train_1d, B_train_1d, L_train = create_pairs_from_split(X_sig_train, y_sig_train, n_tr, augment=True,  seed=42)
A_val_1d,   B_val_1d,   L_val   = create_pairs_from_split(X_sig_val,   y_sig_val,   n_va, augment=False, seed=7)
A_test_1d,  B_test_1d,  L_test  = create_pairs_from_split(X_sig_test,  y_sig_test,  n_te, augment=False, seed=99)

# Labels en (N, 1) pour compatibilité
L_train = L_train.reshape(-1, 1)
L_val   = L_val.reshape(-1, 1)
L_test  = L_test.reshape(-1, 1)

# ── Forme temporelle (N, T, 1)
A_train_time = A_train_1d[..., np.newaxis]
B_train_time = B_train_1d[..., np.newaxis]
A_val_time   = A_val_1d[...,   np.newaxis]
B_val_time   = B_val_1d[...,   np.newaxis]
A_test_time  = A_test_1d[...,  np.newaxis]
B_test_time  = B_test_1d[...,  np.newaxis]

print(f"\n Paires 1D — Train: {A_train_time.shape} | Val: {A_val_time.shape} | Test: {A_test_time.shape}")

# ── Transformation spectrogramme (N, freq, time, 1)
print("Calcul des spectrogrammes pour toutes les paires...")
A_train_spec = transform_to_spectrograms(A_train_1d)
B_train_spec = transform_to_spectrograms(B_train_1d)
A_val_spec   = transform_to_spectrograms(A_val_1d)
B_val_spec   = transform_to_spectrograms(B_val_1d)
A_test_spec  = transform_to_spectrograms(A_test_1d)
B_test_spec  = transform_to_spectrograms(B_test_1d)

SPEC_SHAPE  = A_train_spec.shape[1:]
TIME_SHAPE  = A_train_time.shape[1:]

print(f" Paires 2D spectrogrammes — forme: {A_train_spec.shape}")
print(f"   TIME_SHAPE (Conv1D) : {TIME_SHAPE}")
print(f"   SPEC_SHAPE (Conv2D) : {SPEC_SHAPE}")

for name, L in [("Train", L_train), ("Val", L_val), ("Test", L_test)]:
    n_sim = int(np.sum(L == 0))
    n_dis = int(np.sum(L == 1))
    print(f"   {name:5s}: {len(L):4d} paires  (sim={n_sim}, dis={n_dis})")


Signaux — Train: 195 | Val: 45 | Test: 60

✅ Paires 1D — Train: (8000, 512, 1) | Val: (2000, 512, 1) | Test: (2000, 512, 1)
🔄 Calcul des spectrogrammes pour toutes les paires...
✅ Paires 2D spectrogrammes — forme: (8000, 33, 15, 1)
   TIME_SHAPE (Conv1D) : (512, 1)
   SPEC_SHAPE (Conv2D) : (33, 15, 1)
   Train: 8000 paires  (sim=4000, dis=4000)
   Val  : 2000 paires  (sim=1000, dis=1000)
   Test : 2000 paires  (sim=1000, dis=1000)


## Architecture Hybride: Siamois Conv1D + Conv2D



In [11]:
#  CONTRASTIVE LOSS (margin=1.5, améliorée)
def contrastive_loss(y_true, y_pred):
    """Contrastive Loss — Hadsell et al. (2006).
    y_true : 0 = similaire, 1 = dissimilaire
    y_pred : distance euclidienne ∈ [0, 2] (embeddings L2-normalisés)
    margin = 1.5 (plus strict que 1.0 → meilleure séparation inter-classe)
    """
    margin = MARGIN
    d  = tf.squeeze(tf.cast(y_pred, tf.float32), axis=-1)
    yt = tf.squeeze(tf.cast(y_true, tf.float32), axis=-1)
    loss_pos = (1.0 - yt) * 0.5 * tf.square(d)
    loss_neg = yt          * 0.5 * tf.square(tf.maximum(0.0, margin - d))
    return tf.reduce_mean(loss_pos + loss_neg)

#  ENCODEUR TEMPOREL Conv1D
def build_encoder_1d(input_shape, embed_dim=64, drop_rate=0.3, l2_reg=1e-4):
    """Encodeur Conv1D — capture les patterns temporels.
    Architecture issue du document fourni (section 🔧4), enrichie.
    """
    inp = Input(shape=input_shape, name='time_input')

    # Bloc 1 — large kernel pour tendances basse fréquence
    x = Conv1D(32, 7, padding='same',
               kernel_regularizer=regularizers.l2(l2_reg))(inp)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = MaxPooling1D(2)(x)

    # Bloc 2
    x = Conv1D(64, 5, padding='same',
               kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = SpatialDropout1D(drop_rate * 0.5)(x)
    x = MaxPooling1D(2)(x)

    # Bloc 3
    x = Conv1D(128, 3, padding='same',
               kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = GlobalAveragePooling1D()(x)

    x = Dense(64, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(drop_rate)(x)

    return Model(inp, x, name='encoder_1d')

#  ENCODEUR FRÉQUENTIEL Conv2D
def build_encoder_2d(input_shape, embed_dim=64, drop_rate=0.3, l2_reg=1e-4):
    """Encodeur Conv2D sur spectrogramme — capture la structure temps-fréquence.
    Architecture issue du document fourni (section 🔧5), enrichie.
    """
    inp = Input(shape=input_shape, name='spec_input')

    # Bloc 1
    x = Conv2D(32, (3,3), padding='same', activation='relu',
               kernel_regularizer=regularizers.l2(l2_reg))(inp)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)

    # Bloc 2
    x = Conv2D(64, (3,3), padding='same', activation='relu',
               kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = BatchNormalization()(x)
    x = MaxPooling2D((2,2))(x)

    # Bloc 3
    x = Conv2D(128, (3,3), padding='same', activation='relu',
               kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = BatchNormalization()(x)
    x = GlobalAveragePooling2D()(x)

    x = Dense(64, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(drop_rate)(x)

    return Model(inp, x, name='encoder_2d')

#  ENCODEUR HYBRIDE

def build_hybrid_encoder(time_shape, spec_shape, embed_dim=64,
                          drop_rate=0.3, l2_reg=1e-4):
    """Encodeur hybride : fusionne les représentations temporelle et fréquentielle.
    Architecture issue du document fourni (section 🔧6).

    Entrées : [signal_temps (T,1), spectrogramme (F,T,1)]
    Sortie  : embedding L2-normalisé ∈ ℝ^{embed_dim}
    """
    input_time = Input(shape=time_shape, name='input_time')
    input_spec = Input(shape=spec_shape, name='input_spec')

    # Sous-encodeurs
    enc_1d = build_encoder_1d(time_shape, embed_dim, drop_rate, l2_reg)
    enc_2d = build_encoder_2d(spec_shape, embed_dim, drop_rate, l2_reg)

    emb_time = enc_1d(input_time)   # (batch, 64)
    emb_spec = enc_2d(input_spec)   # (batch, 64)

    # Fusion : Concatenate → Dense → L2-norm
    x = Concatenate()([emb_time, emb_spec])    # (batch, 128)
    x = Dense(128, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = Dropout(drop_rate)(x)
    x = Dense(embed_dim, kernel_regularizer=regularizers.l2(l2_reg))(x)

    # Normalisation L2 → distance ∈ [0, 2]
    embedding = Lambda(
        lambda t: tf.math.l2_normalize(t, axis=1),
        name='l2_norm'
    )(x)

    return Model([input_time, input_spec], embedding, name='hybrid_encoder')

#  RÉSEAU SIAMOIS HYBRIDE COMPLET
def build_siamese_hybrid(time_shape, spec_shape, embed_dim=64,
                          drop_rate=0.3, l2_reg=1e-4):
    """Siamois hybride avec 4 entrées (temps + spec pour chaque branche)."""
    # Encodeur partagé (weight sharing)
    shared_encoder = build_hybrid_encoder(time_shape, spec_shape,
                                          embed_dim, drop_rate, l2_reg)

    # Entrées branche A
    A_time = Input(shape=time_shape, name='A_time')
    A_spec = Input(shape=spec_shape, name='A_spec')
    # Entrées branche B
    B_time = Input(shape=time_shape, name='B_time')
    B_spec = Input(shape=spec_shape, name='B_spec')

    # Embeddings (MÊMES POIDS)
    emb_A = shared_encoder([A_time, A_spec])
    emb_B = shared_encoder([B_time, B_spec])

    # Distance euclidienne ‖emb_A − emb_B‖₂
    distance = Lambda(
        lambda t: tf.norm(t[0] - t[1], axis=1, keepdims=True),
        name='euclidean_distance'
    )([emb_A, emb_B])

    model = Model(
        inputs=[A_time, A_spec, B_time, B_spec],
        outputs=distance,
        name='siamese_hybrid'
    )
    return model, shared_encoder

# ── Instanciation
siamese_model, shared_encoder = build_siamese_hybrid(
    time_shape=TIME_SHAPE,
    spec_shape=SPEC_SHAPE,
    embed_dim=EMBED_DIM,
    drop_rate=DROP_RATE,
    l2_reg=L2_REG
)

siamese_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=3e-4),
    loss=contrastive_loss
)

print("\n" + "="*65)
print("  ARCHITECTURE HYBRIDE — ENCODEUR PARTAGÉ")
print("="*65)
shared_encoder.summary()

print("\n" + "="*65)
print("  RÉSEAU SIAMOIS HYBRIDE COMPLET (4 entrées)")
print("="*65)
siamese_model.summary()
print(f"\n  Paramètres totaux : {siamese_model.count_params():,}")



  ARCHITECTURE HYBRIDE — ENCODEUR PARTAGÉ


Model: "hybrid_encoder"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_time          │ (None, 512, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_spec          │ (None, 33, 15, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_1d          │ (None, 64)        │     44,416 │ input_time[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_2d          │ (None, 64)        │    101,824 │ input_spec[0][0]  │
│ (Functional)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 128)       │          0 │ encoder_1d[0][0], │
│ (Concatenate)       │                   │            │ encoder_2d[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     16,512 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 128)       │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 64)        │      8,256 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ l2_norm (Lambda)    │ (None, 64)        │          0 │ dense_3[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 171,008 (668.00 KB)

 Trainable params: 170,112 (664.50 KB)

 Non-trainable params: 896 (3.50 KB)

... (3 outputs left out)

 Total params: 171,008 (668.00 KB)

 Trainable params: 170,112 (664.50 KB)

 Non-trainable params: 896 (3.50 KB)


  Paramètres totaux : 171,008


## Entraînement du Réseau Siamois

In [12]:
#  CALLBACKS
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=12,
        restore_best_weights=True,
        verbose=1,
        mode='min',
        min_delta=1e-4
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=6,
        min_lr=1e-7,
        verbose=1,
        mode='min'
    )
]

#  ENTRAÎNEMENT HYBRIDE (4 entrées : temps + spec pour A et B)
print(" Entraînement du réseau siamois hybride Conv1D + Conv2D...")
print(f"   Train : {len(L_train)} paires | Val : {len(L_val)} paires")
print(f"   Epochs max : {EPOCHS} | Batch : {BATCH_SIZE} | Margin : {MARGIN}")
print()

history = siamese_model.fit(
    [A_train_time, A_train_spec, B_train_time, B_train_spec],
    L_train,
    validation_data=(
        [A_val_time, A_val_spec, B_val_time, B_val_spec],
        L_val
    ),
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n Entraînement terminé !")
best_epoch    = np.argmin(history.history['val_loss']) + 1
best_val_loss = min(history.history['val_loss'])
final_tr_loss = history.history['loss'][best_epoch - 1]
gap = abs(final_tr_loss - best_val_loss)
print(f"   Meilleur epoch : {best_epoch}")
print(f"   Train loss     : {final_tr_loss:.4f}")
print(f"   Val loss       : {best_val_loss:.4f}")
print(f"   Écart (gap)    : {gap:.4f}  {'OK' if gap < 0.05 else '⚠ Vérifier overfitting'}")


🚀 Entraînement du réseau siamois hybride Conv1D + Conv2D...
   Train : 8000 paires | Val : 2000 paires
   Epochs max : 50 | Batch : 32 | Margin : 1.5

Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 76s 268ms/step - loss: 0.3524 - val_loss: 0.5806 - learning_rate: 3.0000e-04
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 87s 289ms/step - loss: 0.3445 - val_loss: 0.5307 - learning_rate: 3.0000e-04
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 74s 259ms/step - loss: 0.3418 - val_loss: 0.5318 - learning_rate: 3.0000e-04
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 67s 267ms/step - loss: 0.3377 - val_loss: 0.5179 - learning_rate: 3.0000e-04
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 60s 240ms/step - loss: 0.3353 - val_loss: 0.5040 - learning_rate: 3.0000e-04
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 96s 295ms/step - loss: 0.3315 - val_loss: 0.5133 - learning_rate: 3.0000e-04
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 73s 262ms/step - loss: 0.3284 - val_loss: 0.5030 - learning_rate: 3.0000e-04
Epoch 8/50
250/250 ━━━━━━━━━━━

In [13]:
#  COURBES D'APPRENTISSAGE
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
epochs_ran = range(1, len(history.history['loss']) + 1)

ax = axes[0]
ax.plot(epochs_ran, history.history['loss'],     color='royalblue', lw=2, label='Loss Train')
ax.plot(epochs_ran, history.history['val_loss'], color='orangered', lw=2, label='Loss Val', linestyle='--')
ax.axvline(x=best_epoch, color='green', linestyle=':', lw=1.5, label=f'Best epoch ({best_epoch})')
ax.fill_between(epochs_ran, history.history['loss'], history.history['val_loss'],
                alpha=0.15, color='purple', label='Écart overfitting')
ax.set_title('Contrastive Loss — Train vs Validation', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Contrastive Loss')
ax.legend(); ax.grid(True, linestyle='--', alpha=0.5)
ax.set_xlim(1, len(epochs_ran))

ax2 = axes[1]
ratio = [v/max(t,1e-8) for t,v in zip(history.history['loss'], history.history['val_loss'])]
ax2.plot(epochs_ran, ratio, color='purple', lw=2)
ax2.axhline(y=1.0, color='black', linestyle='--', lw=1, label='Ratio = 1 (idéal)')
ax2.axhline(y=1.3, color='red',   linestyle=':', lw=1, label='Seuil overfitting = 1.3')
ax2.fill_between(epochs_ran, 1.3, [max(r, 1.3) for r in ratio], alpha=0.2, color='red')
ax2.set_title('Ratio Val/Train Loss (diagnostic)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Val / Train Loss')
ax2.legend(); ax2.grid(True, linestyle='--', alpha=0.5)
ax2.set_xlim(1, len(epochs_ran))

plt.tight_layout()
plt.savefig('courbes_apprentissage_hybride.png', dpi=150, bbox_inches='tight')
plt.show()
print("Courbes d'apprentissage sauvegardées.")


✅ Courbes d'apprentissage sauvegardées.

🔴 Diagnostic : Overfitting (ratio > 1.5)


---
# 📊 Section 8 — Évaluation des Performances
---

In [14]:
#  PRÉDICTION SUR LE SET DE TEST (modèle hybride)
print(" Calcul des distances")
distances_test = siamese_model.predict(
    [A_test_time, A_test_spec, B_test_time, B_test_spec],
    verbose=0
).flatten()

y_true_test = L_test.flatten().astype(int)

print(f"   Distances — min: {distances_test.min():.4f} | max: {distances_test.max():.4f} | mean: {distances_test.mean():.4f}")
print(f"   Embeddings L2-normalisés → distance ∈ [0, 2], MARGIN={MARGIN}")

d_sim = distances_test[y_true_test == 0]
d_dis = distances_test[y_true_test == 1]
sep   = d_dis.mean() - d_sim.mean()
print(f"\n   Paires similaires    : mean={d_sim.mean():.4f} ± {d_sim.std():.4f}")
print(f"   Paires dissimilaires : mean={d_dis.mean():.4f} ± {d_dis.std():.4f}")
print(f"   Séparation Δmean     : {sep:.4f}  {' Bonne séparation' if sep > 0.4 else '⚠ Séparation faible'}")

#  SEUIL OPTIMAL PAR ROC
fpr, tpr, thresholds = roc_curve(y_true_test, distances_test)
roc_auc    = auc(fpr, tpr)
youden_idx = np.argmax(tpr - fpr)
optimal_thr = thresholds[youden_idx]

print(f"\n AUC-ROC        : {roc_auc:.4f}  {'✅' if roc_auc > 0.85 else '⚠'}")
print(f"   Seuil optimal  : {optimal_thr:.4f}  (indice Youden)")
print(f"   TPR optimal    : {tpr[youden_idx]:.4f}")
print(f"   FPR optimal    : {fpr[youden_idx]:.4f}")


📐 Calcul des distances (modèle hybride) sur le test set...
   Distances — min: 0.0024 | max: 1.5539 | mean: 0.6291
   Embeddings L2-normalisés → distance ∈ [0, 2], MARGIN=1.5

   Paires similaires    : mean=0.2774 ± 0.3818
   Paires dissimilaires : mean=0.9808 ± 0.5841
   Séparation Δmean     : 0.7034  ✅ Bonne séparation

🎯 AUC-ROC        : 0.8387  ⚠
   Seuil optimal  : 0.6531  (indice Youden)
   TPR optimal    : 0.6890
   FPR optimal    : 0.1570


In [15]:
#  MÉTRIQUES DE CLASSIFICATION
y_pred_test = (distances_test >= optimal_thr).astype(int)

acc    = accuracy_score(y_true_test, y_pred_test)
f1     = f1_score(y_true_test, y_pred_test, average='binary')
cm     = confusion_matrix(y_true_test, y_pred_test)

tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # Recall / TPR
specificity = tn / (tn + fp)  # TNR
precision   = tp / (tp + fp) if (tp + fp) > 0 else 0

print("\n" + "="*55)
print("  RAPPORT DE PERFORMANCES — RÉSEAU SIAMOIS CONV1D")
print("="*55)
print(f"  Accuracy    : {acc*100:.2f}%")
print(f"  F1-Score    : {f1:.4f}")
print(f"  AUC-ROC     : {roc_auc:.4f}")
print(f"  Sensibilité : {sensitivity:.4f}  (TPR)")
print(f"  Spécificité : {specificity:.4f}  (TNR)")
print(f"  Précision   : {precision:.4f}")
print("="*55)
print()
print(classification_report(
    y_true_test, y_pred_test,
    target_names=['Similaire (0)', 'Dissimilaire (1)']
))


  RAPPORT DE PERFORMANCES — RÉSEAU SIAMOIS CONV1D
  Accuracy    : 76.60%
  F1-Score    : 0.7465
  AUC-ROC     : 0.8387
  Sensibilité : 0.6890  (TPR)
  Spécificité : 0.8430  (TNR)
  Précision   : 0.8144

                  precision    recall  f1-score   support

   Similaire (0)       0.73      0.84      0.78      1000
Dissimilaire (1)       0.81      0.69      0.75      1000

        accuracy                           0.77      2000
       macro avg       0.77      0.77      0.76      2000
    weighted avg       0.77      0.77      0.76      2000



In [16]:

#  VISUALISATION COMPLÈTE DES MÉTRIQUES
fig = plt.figure(figsize=(16, 13))
fig.suptitle("Évaluation",
             fontsize=14, fontweight='bold', y=0.98)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.32)

# Matrice de confusion
ax1 = fig.add_subplot(gs[0, 0])
im = ax1.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax1, fraction=0.046)
cls_names = ['Similaire\n(y=0)', 'Dissimilaire\n(y=1)']
ax1.set_xticks([0,1]); ax1.set_xticklabels(cls_names, fontsize=10)
ax1.set_yticks([0,1]); ax1.set_yticklabels(cls_names, fontsize=10, rotation=90, va='center')
thresh_cm = cm.max() / 2
for i in range(2):
    for j in range(2):
        ax1.text(j, i, f'{cm[i,j]}\n({cm[i,j]/cm.sum()*100:.1f}%)',
                 ha='center', va='center', fontsize=13, fontweight='bold',
                 color='white' if cm[i,j] > thresh_cm else 'black')
ax1.set_xlabel('Prédit', fontsize=11); ax1.set_ylabel('Réel', fontsize=11)
ax1.set_title('Matrice de Confusion', fontsize=12, fontweight='bold')

# Courbe ROC
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(fpr, tpr, color='royalblue', lw=2.5, label=f'ROC Hybride (AUC={roc_auc:.4f})')
ax2.plot([0,1],[0,1],'k--', lw=1.5, label='Aléatoire (AUC=0.50)')
ax2.scatter(fpr[youden_idx], tpr[youden_idx], color='red', s=100, zorder=5,
            label=f'Seuil optimal = {optimal_thr:.3f}')
ax2.fill_between(fpr, tpr, alpha=0.1, color='royalblue')
ax2.set_xlabel('FPR', fontsize=11); ax2.set_ylabel('TPR', fontsize=11)
ax2.set_title('Courbe ROC', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9); ax2.grid(True, linestyle='--', alpha=0.4)
ax2.set_xlim(-0.01, 1.01); ax2.set_ylim(-0.01, 1.05)

# Distribution des distances
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(distances_test[y_true_test==0], bins=30, alpha=0.7, color='steelblue',
         label='Similaires (y=0)', density=True)
ax3.hist(distances_test[y_true_test==1], bins=30, alpha=0.7, color='tomato',
         label='Dissimilaires (y=1)', density=True)
ax3.axvline(x=optimal_thr, color='green', linestyle='--', lw=2,
            label=f'Seuil = {optimal_thr:.3f}')
ax3.axvline(x=MARGIN, color='orange', linestyle=':', lw=1.5,
            label=f'Margin = {MARGIN}')
ax3.set_xlabel('Distance Euclidienne', fontsize=11)
ax3.set_ylabel('Densité', fontsize=11)
ax3.set_title('Distribution des Distances (Test)', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9); ax3.grid(True, linestyle='--', alpha=0.4)

#Résumé métriques
ax4 = fig.add_subplot(gs[1, 1])
metric_names  = ['Accuracy', 'F1-Score', 'AUC-ROC', 'Sensibilité', 'Spécificité', 'Précision']
metric_values = [acc, f1, roc_auc, sensitivity, specificity, precision]
colors_bar    = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336','#00BCD4']
bars = ax4.barh(metric_names, metric_values, color=colors_bar, alpha=0.85, edgecolor='white', linewidth=1.2)
ax4.set_xlim(0, 1.15)
for bar, val in zip(bars, metric_values):
    ax4.text(val+0.01, bar.get_y()+bar.get_height()/2,
             f'{val:.4f}', va='center', fontsize=10, fontweight='bold')
ax4.axvline(x=1.0, color='black', linestyle='--', lw=1, alpha=0.4)
ax4.set_xlabel('Score', fontsize=11)
ax4.set_title('Résumé des Métriques', fontsize=12, fontweight='bold')
ax4.grid(True, axis='x', linestyle='--', alpha=0.4)

plt.savefig('evaluation_siamois_hybride.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Figure d'évaluation complète sauvegardée.")


✅ Figure d'évaluation complète sauvegardée.


##Visualisation de l'Espace d'Embedding (t-SNE)



In [17]:
#  VISUALISATION t-SNE DE L'ESPACE D'EMBEDDING
from sklearn.manifold import TSNE

# Préparer tous les signaux pour l'extraction d'embeddings
X_all_time = X_all[..., np.newaxis]
X_all_spec = transform_to_spectrograms(X_all)

print(f"Extraction des embeddings hybrides pour {len(X_all)} signaux...")
embeddings_all = shared_encoder.predict(
    [X_all_time, X_all_spec], verbose=0
)

print(f" Embeddings extraits : {embeddings_all.shape}")

# t-SNE 2D
print("Calcul t-SNE.")
tsne  = TSNE(n_components=2, perplexity=20, random_state=42, n_iter=1000)
emb_2d = tsne.fit_transform(embeddings_all)

# Visualisation
fig, ax = plt.subplots(1, 1, figsize=(10, 7))
class_names   = ['Normal', 'IRF (Inner Race Fault)', 'ORF (Outer Race Fault)']
class_colors  = ['#2196F3', '#F44336', '#4CAF50']
class_markers = ['o', 's', '^']

for cls_idx, (cname, ccolor, cmark) in enumerate(zip(class_names, class_colors, class_markers)):
    mask = (y_all == cls_idx)
    ax.scatter(emb_2d[mask,0], emb_2d[mask,1],
               c=ccolor, label=cname, marker=cmark,
               s=80, alpha=0.85, edgecolors='white', linewidths=0.5)

ax.set_title("Espace d'Embedding t-SNE ",
             fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE Dim 1'); ax.set_ylabel('t-SNE Dim 2')
ax.legend(fontsize=11, framealpha=0.9)
ax.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.savefig('tsne_embedding_hybride.png', dpi=150, bbox_inches='tight')
plt.show()
print(" t-SNE sauvegardé. Des clusters bien séparés → bon apprentissage métrique.")


Extraction des embeddings hybrides pour 300 signaux...
✅ Embeddings extraits : (300, 64)
Calcul t-SNE (peut prendre quelques secondes)...
✅ t-SNE sauvegardé. Des clusters bien séparés → bon apprentissage métrique.


## Classification Multi-classe par k-NN sur les Embeddings


In [18]:
#  CLASSIFICATION k-NN SUR LES EMBEDDINGS HYBRIDES

from sklearn.neighbors import KNeighborsClassifier

# Utiliser les splits de signaux déjà définis (X_sig_train/test)
X_s_tr_time = X_sig_train[..., np.newaxis]
X_s_te_time = X_sig_test[...,  np.newaxis]
X_s_tr_spec = transform_to_spectrograms(X_sig_train)
X_s_te_spec = transform_to_spectrograms(X_sig_test)

emb_knn_train = shared_encoder.predict([X_s_tr_time, X_s_tr_spec], verbose=0)
emb_knn_test  = shared_encoder.predict([X_s_te_time, X_s_te_spec], verbose=0)

# k-NN (k=5, distance euclidienne)
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(emb_knn_train, y_sig_train)
y_pred_knn = knn.predict(emb_knn_test)

acc_knn = accuracy_score(y_sig_test, y_pred_knn)
f1_knn  = f1_score(y_sig_test, y_pred_knn, average='macro')
cm_knn  = confusion_matrix(y_sig_test, y_pred_knn)

print("\n" + "="*55)
print("  CLASSIFICATION k-NN (k=5) SUR EMBEDDINGS HYBRIDES")
print("="*55)
print(f"  Accuracy  : {acc_knn*100:.2f}%")
print(f"  F1 (macro): {f1_knn:.4f}")
print()
print(classification_report(y_sig_test, y_pred_knn,
                             target_names=['Normal','IRF','ORF']))

# Matrice de confusion multi-classe
fig, ax = plt.subplots(figsize=(7, 5))
im2 = ax.imshow(cm_knn, cmap='YlOrRd', interpolation='nearest')
fig.colorbar(im2, ax=ax)
tick_labels = ['Normal','IRF','ORF']
ax.set_xticks([0,1,2]); ax.set_xticklabels(tick_labels, fontsize=11)
ax.set_yticks([0,1,2]); ax.set_yticklabels(tick_labels, fontsize=11, rotation=90, va='center')
thresh2 = cm_knn.max() / 2
for i in range(3):
    for j in range(3):
        ax.text(j, i, str(cm_knn[i,j]), ha='center', va='center', fontsize=13,
                fontweight='bold', color='white' if cm_knn[i,j]>thresh2 else 'black')
ax.set_title('Matrice de Confusion — k-NN sur Embeddings Hybrides\n(3 classes)', fontsize=12, fontweight='bold')
ax.set_xlabel('Classe Prédite', fontsize=11); ax.set_ylabel('Classe Réelle', fontsize=11)
plt.tight_layout()
plt.savefig('confusion_knn_hybride.png', dpi=150, bbox_inches='tight')
plt.show()
print(" Matrice de confusion multi-classe sauvegardée.")



  CLASSIFICATION k-NN (k=5) SUR EMBEDDINGS HYBRIDES
  Accuracy  : 78.33%
  F1 (macro): 0.7863

              precision    recall  f1-score   support

      Normal       0.72      0.65      0.68        20
         IRF       0.67      0.80      0.73        20
         ORF       1.00      0.90      0.95        20

    accuracy                           0.78        60
   macro avg       0.80      0.78      0.79        60
weighted avg       0.80      0.78      0.79        60

✅ Matrice de confusion multi-classe sauvegardée.


## Analyse



In [23]:

#  ANALYSE COMPARATIVE — AMÉLIORATIONS APPLIQUÉES


print("\n RÉSULTATS FINAUX (Test) :")
print(f"   AUC-ROC             : {roc_auc:.4f}  {'✅' if roc_auc > 0.85 else '⚠'}")
print(f"   Accuracy (binaire)  : {acc*100:.2f}%")
print(f"   F1-Score (binaire)  : {f1:.4f}")
print(f"   Précision           : {precision:.4f}")
print(f"   Sensibilité         : {sensitivity:.4f}")
print(f"   Spécificité         : {specificity:.4f}")
print(f"   Accuracy k-NN (3cls): {acc_knn*100:.2f}%")
print(f"   F1 macro k-NN       : {f1_knn:.4f}")





 RÉSULTATS FINAUX (Test) :
   AUC-ROC             : 0.8387  ⚠
   Accuracy (binaire)  : 76.60%
   F1-Score (binaire)  : 0.7465
   Précision           : 0.8144
   Sensibilité         : 0.6890
   Spécificité         : 0.8430
   Accuracy k-NN (3cls): 78.33%
   F1 macro k-NN       : 0.7863


#Conclusion


In [22]:
#  RÉSUMÉ FINAL — MODÈLE HYBRIDE SIAMOIS Conv1D + Conv2D

print(f"  PERFORMANCES BINAIRES (Test)   " )
print(f"  Accuracy     : {acc*100:6.2f}%  ")
print(f"  F1-Score     : {f1:.4f}          ")
print(f"  AUC-ROC      : {roc_auc:.4f}      ")
print(f"  Sensibilité  : {sensitivity:.4f}    ")
print(f"  Spécificité  : {specificity:.4f}      ")
print( "═"*58 )
print(f"  CLASSIFICATION MULTI-CLASSE k-NN ")
print(f"  Accuracy k-NN  : {acc_knn*100:6.2f}%")
print(f"  F1 macro k-NN  : {f1_knn:.4f}         ")


  PERFORMANCES BINAIRES (Test)   
  Accuracy     :  76.60%  
  F1-Score     : 0.7465          
  AUC-ROC      : 0.8387      
  Sensibilité  : 0.6890    
  Spécificité  : 0.8430      
══════════════════════════════════════════════════════════
  CLASSIFICATION MULTI-CLASSE k-NN 
  Accuracy k-NN  :  78.33%
  F1 macro k-NN  : 0.7863         
